In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
완전한 통합 3D Keypoint 생성 및 시각화 (Cross-ratio 검증 + JSON 저장 버전)
"""

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy.spatial import ConvexHull, cKDTree
from scipy.spatial.distance import cdist
from pathlib import Path
from itertools import combinations

try:
    import open3d as o3d
    HAS_OPEN3D = True
except ImportError:
    HAS_OPEN3D = False

import sys
_SUB = Path.cwd() if (Path.cwd() / 'process_models.py').exists() else Path.cwd() / 'sub'
sys.path.insert(0, str(_SUB))
from compute_symmetry_centers import symmetry_axis
from process_models import PALIGNED_VALUES

# 대칭축이 하나인 물체는 대칭축 좌표계(세로 = 대칭축, 원점 = 축 위)에서도 최외각 상자를 만들어,
# 대칭 회전 후 키포인트 어긋남이 모델 좌표계 상자보다 작으면 그 상자를 쓴다. False면 모델 좌표계 기준만 (이전 방식).
AXIS_BOX = True

# ==================== 1. 분할 설정 ====================
SPLIT_CONFIG = {
    1: {'type': 'z_zero'}, 2: {'type': 'z_zero'}, 3: {'type': 'z_zero'},
    4: {'type': 'z_zero'}, 5: {'type': 'z_zero'}, 6: {'type': 'z_zero'},
    9: {'type': 'z_zero'}, 11: {'type': 'z_zero'}, 12: {'type': 'z_zero'},
    13: {'type': 'z_zero'}, 14: {'type': 'z_zero'}, 16: {'type': 'z_zero'},
    21: {'type': 'z_zero'},
    7: {'type': 'custom_line', 'point1': (60, 30), 'point2': (-60, -40)},
    8: {'type': 'custom_line', 'point1': (-5, 25), 'point2': (5, -25)},
    10: {'type': 'custom_line', 'point1': (60, -10), 'point2': (-60, 30)},
    15: {'type': 'custom_line', 'point1': (60, 0), 'point2': (-60, 0)},
    17: {'type': 'custom_line', 'point1': (-60, 30), 'point2': (60, -10)},
    18: {'type': 'custom_line', 'point1': (60, -10), 'point2': (-60, 30)},
    19: {'type': 'custom_line', 'point1': (-60, 0), 'point2': (60, 0)},
    20: {'type': 'custom_line', 'point1': (0, 30), 'point2': (0, -30)},
}

# EDGE_OVERRIDE는 필요시 여기에 정의
EDGE_OVERRIDE = {}

# ==================== 2. PLY 로드 ====================
def load_points_from_ply(file_path):
    if not HAS_OPEN3D:
        return None
    try:
        # Surface sampling (face-aware) instead of vertex-only read_point_cloud.
        # For low-poly meshes (e.g. obj_25=466, obj_27=920) the vertex set is too
        # sparse for reliable axis-aligned corner detection downstream.
        mesh = o3d.io.read_triangle_mesh(str(file_path))
        if mesh.has_triangles():
            o3d.utility.random.seed(0)
            pcd = mesh.sample_points_uniformly(number_of_points=5000)
        else:
            pcd = o3d.io.read_point_cloud(str(file_path))
        points = np.asarray(pcd.points, dtype=np.float32)
        return points if len(points) > 0 else None
    except:
        return None

# ==================== 3. 키포인트 생성 ====================
def compute_2d_minimal_bbox(points):
    if len(points) < 3:
        return None
    
    xy_points = points[:, :2]
    z_min, z_max = points[:, 2].min(), points[:, 2].max()
    
    try:
        hull_2d = ConvexHull(xy_points)
        hull_points = xy_points[hull_2d.vertices]
    except:
        x_min, x_max = xy_points[:, 0].min(), xy_points[:, 0].max()
        y_min, y_max = xy_points[:, 1].min(), xy_points[:, 1].max()
        best_rect = np.array([[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]])
    else:
        min_area = float('inf')
        best_rect = None
        angles = np.linspace(0, np.pi, 180)
        
        for angle in angles:
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            R = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
            rotated_hull = hull_points @ R.T
            x_min, x_max = rotated_hull[:, 0].min(), rotated_hull[:, 0].max()
            y_min, y_max = rotated_hull[:, 1].min(), rotated_hull[:, 1].max()
            area = (x_max - x_min) * (y_max - y_min)
            
            if area < min_area:
                min_area = area
                rect_corners_rot = np.array([[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]])
                best_rect = rect_corners_rot @ R
    
    corners_3d = np.array([
        [best_rect[0][0], best_rect[0][1], z_min],
        [best_rect[1][0], best_rect[1][1], z_min],
        [best_rect[2][0], best_rect[2][1], z_min],
        [best_rect[3][0], best_rect[3][1], z_min],
        [best_rect[0][0], best_rect[0][1], z_max],
        [best_rect[1][0], best_rect[1][1], z_max],
        [best_rect[2][0], best_rect[2][1], z_max],
        [best_rect[3][0], best_rect[3][1], z_max],
    ], dtype=np.float64)
    
    return corners_3d

def compute_aligned_bbox(points):
    """8 corners of the box aligned with the x, y, z axes (outermost points)."""
    lo, hi = points.min(0), points.max(0)
    rect = [(lo[0], lo[1]), (hi[0], lo[1]), (hi[0], hi[1]), (lo[0], hi[1])]
    return np.array([[x, y, z] for z in (lo[2], hi[2]) for x, y in rect], dtype=np.float64)

def generate_keypoints_32(corners):
    """32개 키포인트 생성: 0-7 corners, 8-31 edge midpoints"""
    keypoints = list(corners)
    edges = [
        (0,1), (1,2), (2,3), (3,0),  # bottom
        (4,5), (5,6), (6,7), (7,4),  # top
        (0,4), (1,5), (2,6), (3,7),  # vertical
    ]
    
    for (i0, i1) in edges:
        A, B = corners[i0], corners[i1]
        keypoints.append((2 * A + 1 * B) / 3)
        keypoints.append((1 * A + 2 * B) / 3)
    
    return np.array(keypoints, dtype=np.float64)

# ==================== 4. Upper/Lower 분할 ====================
def split_by_z_zero(keypoints):
    upper_mask = keypoints[:, 2] > 0
    lower_mask = keypoints[:, 2] <= 0
    return upper_mask, lower_mask

def split_by_custom_line(keypoints, point1, point2):
    x1, y1 = point1
    x2, y2 = point2
    a = y2 - y1
    b = -(x2 - x1)
    c = (x2 - x1) * y1 - (y2 - y1) * x1
    
    xy_coords = keypoints[:, :2]
    distances = a * xy_coords[:, 0] + b * xy_coords[:, 1] + c
    upper_mask = distances > 0
    lower_mask = distances <= 0
    return upper_mask, lower_mask

# ==================== 5. Corners 추적 재배열 ====================
def rearrange_with_corners(keypoints_32, mask):
    corner_mask = np.zeros(32, dtype=bool)
    corner_mask[:8] = True
    
    selected_corners_mask = mask & corner_mask
    selected_non_corners_mask = mask & ~corner_mask
    
    corners = keypoints_32[selected_corners_mask]
    non_corners = keypoints_32[selected_non_corners_mask]
    
    n_corners = len(corners)
    n_remaining = 16 - n_corners
    
    if n_remaining <= 0:
        result = corners[:16]
        corner_indices = list(range(min(n_corners, 16)))
    else:
        if len(non_corners) >= n_remaining:
            indices = np.linspace(0, len(non_corners) - 1, n_remaining, dtype=int)
            selected_non = non_corners[indices]
        elif len(non_corners) > 0:
            padding = np.zeros((n_remaining - len(non_corners), 3))
            selected_non = np.vstack([non_corners, padding])
        else:
            selected_non = np.zeros((n_remaining, 3))
        
        result = np.vstack([corners, selected_non])
        corner_indices = list(range(n_corners))
    
    return result, corner_indices

# ==================== 6. Cross-ratio 계산 ====================
def compute_cross_ratio(points):
    """4개 점의 cross-ratio: CR(A,B,C,D) = (AC*BD)/(AD*BC)"""
    if len(points) != 4:
        return None
    
    A, B, C, D = points
    
    AC = np.linalg.norm(C - A)
    BD = np.linalg.norm(D - B)
    AD = np.linalg.norm(D - A)
    BC = np.linalg.norm(C - B)
    
    if AD < 1e-10 or BC < 1e-10:
        return None
    
    cr = (AC * BD) / (AD * BC)
    return cr

# ==================== 7. Edges Mask 재매핑 (Cross-ratio) ====================
def find_corner_edges(corner_points, corner_indices):
    """거리 기반으로 corners 간의 edges 찾기"""
    n = len(corner_points)
    if n < 2:
        return []
    
    distances = cdist(corner_points, corner_points)
    edges = []
    used = set()
    
    for i in range(n):
        dists = [(j, distances[i][j]) for j in range(n) if j != i]
        dists.sort(key=lambda x: x[1])
        
        for j, d in dists[:2]:
            edge = tuple(sorted([corner_indices[i], corner_indices[j]]))
            if edge not in used:
                edges.append((corner_indices[i], corner_indices[j]))
                used.add(edge)
                if len(edges) >= 4:
                    return edges[:4]
    
    return edges[:4]

def find_edge_midpoints_cross_ratio(total_kpts, c1, c2, used_points):
    """Cross-ratio 체크로 완벽한 중간점 찾기"""
    p1, p2 = total_kpts[c1], total_kpts[c2]
    line_vec = p2 - p1
    line_len = np.linalg.norm(line_vec)
    
    if line_len < 1e-6:
        return [c1, c1, c2, c2]
    
    # 직선에 가까운 후보점들
    candidates = []
    for i in range(len(total_kpts)):
        if i in used_points or i == c1 or i == c2:
            continue
        
        point = total_kpts[i]
        vec = point - p1
        proj_len = np.dot(vec, line_vec) / line_len
        
        if 0 <= proj_len <= line_len:
            proj_point = p1 + (proj_len / line_len) * line_vec
            dist = np.linalg.norm(point - proj_point)
            candidates.append((i, dist, proj_len))
    
    if len(candidates) < 2:
        # 후보 부족 - fallback
        candidates.sort(key=lambda x: x[1])
        mids = [x[0] for x in candidates] if candidates else []
        all_pts = [c1] + mids + [c2]
        while len(all_pts) < 4:
            all_pts.insert(-1, c2)
        return all_pts[:4]
    
    # Cross-ratio 기반 최적 조합 찾기
    target_cr = 4.0 / 3.0
    best_combo = None
    best_error = float('inf')
    
    candidates.sort(key=lambda x: x[1])
    top_candidates = candidates[:min(15, len(candidates))]
    
    # 2개 중간점 조합 시도
    for combo in combinations(top_candidates, 2):
        mid_indices = [x[0] for x in combo]
        all_pts = [c1] + mid_indices + [c2]
        
        # 위치순 정렬
        all_pts.sort(key=lambda idx: np.dot(
            total_kpts[idx] - p1, line_vec
        ) / line_len if line_len > 0 else 0)
        
        if len(all_pts) == 4:
            points_3d = total_kpts[all_pts]
            cr = compute_cross_ratio(points_3d)
            
            if cr is not None:
                error = abs(cr - target_cr)
                
                if error < best_error:
                    best_error = error
                    best_combo = all_pts
    
    # 좋은 조합 발견 (5% 오차 이내)
    if best_combo is not None and best_error < 0.067:  # 4/3 * 0.05
        return best_combo
    
    # Fallback: 거리순
    candidates.sort(key=lambda x: x[1])
    mids = [x[0] for x in candidates[:2]]
    all_pts = [c1] + mids + [c2]
    all_pts.sort(key=lambda idx: np.dot(
        total_kpts[idx] - p1, line_vec
    ) / line_len if line_len > 0 else 0)
    
    while len(all_pts) < 4:
        all_pts.insert(-1, c2)
    
    return all_pts[:4]

def create_manual_edges_mask(upper_kpts, lower_kpts, upper_corners, lower_corners, edge_config):
    total_kpts = np.vstack([upper_kpts, lower_kpts])
    n_upper = len(upper_kpts)
    
    edges_mask = {}
    used_points = set()
    
    # Upper edges
    if 'upper_edges' in edge_config:
        for i, (c1_idx, c2_idx) in enumerate(edge_config['upper_edges']):
            if c1_idx < len(upper_corners) and c2_idx < len(upper_corners):
                c1 = upper_corners[c1_idx]
                c2 = upper_corners[c2_idx]
                edge_pts = find_edge_midpoints_cross_ratio(total_kpts, c1, c2, used_points)
                if len(edge_pts) == 4:
                    edges_mask[f'upper_edge_{i+1}'] = edge_pts
                    for pt in edge_pts[1:-1]:
                        used_points.add(pt)
    
    # Lower edges
    lower_corners_adj = [idx + n_upper for idx in lower_corners]
    if 'lower_edges' in edge_config:
        for i, (c1_idx, c2_idx) in enumerate(edge_config['lower_edges']):
            if c1_idx < len(lower_corners) and c2_idx < len(lower_corners):
                c1 = lower_corners_adj[c1_idx]
                c2 = lower_corners_adj[c2_idx]
                edge_pts = find_edge_midpoints_cross_ratio(total_kpts, c1, c2, used_points)
                if len(edge_pts) == 4:
                    edges_mask[f'lower_edge_{i+1}'] = edge_pts
                    for pt in edge_pts[1:-1]:
                        used_points.add(pt)
    
    # Vertical edges
    if 'vertical_edges' in edge_config:
        for i, (u_idx, l_idx) in enumerate(edge_config['vertical_edges']):
            if u_idx < len(upper_corners) and l_idx < len(lower_corners):
                c1 = upper_corners[u_idx]
                c2 = lower_corners_adj[l_idx]
                edge_pts = find_edge_midpoints_cross_ratio(total_kpts, c1, c2, used_points)
                if len(edge_pts) == 4:
                    edges_mask[f'vertical_edge_{i+1}'] = edge_pts
                    for pt in edge_pts[1:-1]:
                        used_points.add(pt)
    
    return edges_mask

def create_edges_mask_cross_ratio(upper_kpts, lower_kpts, upper_corners, lower_corners):
    """Cross-ratio 검증으로 Edges mask 생성"""
    total_kpts = np.vstack([upper_kpts, lower_kpts])
    n_upper = len(upper_kpts)
    
    lower_corners_adj = [idx + n_upper for idx in lower_corners]
    
    edges_mask = {}
    used_points = set()
    
    # Upper edges
    if len(upper_corners) >= 2:
        upper_corner_pts = upper_kpts[upper_corners]
        upper_edge_pairs = find_corner_edges(upper_corner_pts, upper_corners)
        
        for i, (c1, c2) in enumerate(upper_edge_pairs):
            edge_pts = find_edge_midpoints_cross_ratio(total_kpts, c1, c2, used_points)
            if len(edge_pts) == 4:
                edges_mask[f'upper_edge_{i+1}'] = edge_pts
                for pt in edge_pts[1:-1]:
                    used_points.add(pt)
    
    # Lower edges
    if len(lower_corners) >= 2:
        lower_corner_pts = lower_kpts[lower_corners]
        lower_edge_pairs = find_corner_edges(lower_corner_pts, lower_corners_adj)
        
        for i, (c1, c2) in enumerate(lower_edge_pairs):
            edge_pts = find_edge_midpoints_cross_ratio(total_kpts, c1, c2, used_points)
            if len(edge_pts) == 4:
                edges_mask[f'lower_edge_{i+1}'] = edge_pts
                for pt in edge_pts[1:-1]:
                    used_points.add(pt)
    
    # Vertical edges
    n_vert = min(len(upper_corners), len(lower_corners))
    for i in range(n_vert):
        c1 = upper_corners[i]
        c2 = lower_corners_adj[i]
        edge_pts = find_edge_midpoints_cross_ratio(total_kpts, c1, c2, used_points)
        if len(edge_pts) == 4:
            edges_mask[f'vertical_edge_{i+1}'] = edge_pts
            for pt in edge_pts[1:-1]:
                used_points.add(pt)
    
    return edges_mask

# ==================== 8. 파이프라인 ====================
def _rotation_axis_angle(R):
    angle = np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1.0, 1.0)))
    ev, evec = np.linalg.eig(R)
    axis = np.real(evec[:, np.argmin(np.abs(ev - 1))])
    return axis / np.linalg.norm(axis), angle

def _axes_frame(z, x):
    x = x - z * (x @ z)
    x /= np.linalg.norm(x)
    return np.stack([x, np.cross(z, x), z], 1)

def symmetry_axis_frame(obj_id, ply_path):
    """Symmetry data of an object in the frame of ply_path: (point, Q, transforms, aligned), or None without symmetry.

    point, Q: origin and axes (columns) of the symmetry frame, None if it cannot be built.
      - one symmetry axis: z = the axis; the box rectangle is the minimum-area one (aligned False)
      - several axes (e.g. YCB-V 16, ITODD 3/11/19): z = the axis of the highest order (closest to the model z on a
        tie), x = a perpendicular 180-degree axis; the box is aligned with these axes (aligned True), choosing the x
        that gives the smallest rectangle
    transforms: 4x4 symmetries for the keypoint check (continuous: 90/180/270 deg about the axis).
    models_info is in the original BOP frame; objects in PALIGNED_VALUES (meshes moved by -P) are converted to the
    moved frame, so the PLY folder must hold the meshes in the training frame (<dataset>/models_eval).
    """
    info_path = Path(ply_path).parent / 'models_info.json'
    if not info_path.exists():
        return None
    info = json.loads(info_path.read_text()).get(str(obj_id))
    found = symmetry_axis(info) if info else None
    if found is None:
        return None
    dataset = Path(ply_path).resolve().parent.parent.name
    P = np.asarray(PALIGNED_VALUES.get(dataset, {}).get(obj_id, [0.0, 0.0, 0.0]))
    T_in, T_out = np.eye(4), np.eye(4)
    T_in[:3, 3], T_out[:3, 3] = P, -P
    point, axis, _ = found
    if info.get('symmetries_continuous'):
        mats = []
        for deg in (90, 180, 270):
            a = np.deg2rad(deg); K = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
            M = np.eye(4); M[:3, :3] = np.eye(3) + np.sin(a) * K + (1 - np.cos(a)) * K @ K
            M[:3, 3] = point - M[:3, :3] @ point
            mats.append(M)
    else:
        mats = [np.asarray(m, dtype=np.float64).reshape(4, 4) for m in info['symmetries_discrete']]
    transforms = [T_out @ M @ T_in for M in mats]
    point = point - P
    if axis is not None:
        k = int(np.argmax(np.abs(axis)))
        if axis[k] < 0:
            axis = -axis
        point = point - axis * (point @ axis)
        return point, _axes_frame(axis, np.eye(3)[0] if k != 0 else np.eye(3)[1]), transforms, False
    rot = [_rotation_axis_angle(M[:3, :3]) for M in transforms]
    smallest = min(a for _, a in rot)
    z_candidates = [ax for ax, a in rot if abs(a - smallest) < 1.0]
    z = max(z_candidates, key=lambda ax: abs(ax[2]))
    z = z * np.sign(z[np.argmax(np.abs(z))])
    x_candidates = [ax for ax, a in rot if abs(a - 180) < 1.0 and abs(ax @ z) < 0.1]
    if not x_candidates:
        return None, None, transforms, False
    points = load_points_from_ply(ply_path)
    def rect_area(x):
        local = (points - point) @ _axes_frame(z, x)
        return np.prod(local[:, :2].max(0) - local[:, :2].min(0))
    return point, _axes_frame(z, min(x_candidates, key=rect_area)), transforms, True

def keypoint_symmetry_error(keypoints, transforms):
    """Largest distance (mm) from a symmetry-transformed keypoint to the nearest keypoint."""
    if not transforms:
        return 0.0
    tree = cKDTree(keypoints)
    return max(tree.query(keypoints @ M[:3, :3].T + M[:3, 3])[0].max() for M in transforms)

def _valid(result):
    return (result is not None and result['avg_cr_error'] < 1e-3 and len(result['edges_mask']) == 12
            and all(len(e) == 4 for e in result['edges_mask'].values()))

def process_single_object(obj_id, ply_path):
    points = load_points_from_ply(ply_path)
    if points is None:
        return None
    result = _process_points(obj_id, points)
    if result is None:
        return None
    sym = symmetry_axis_frame(obj_id, ply_path)
    transforms = sym[2] if sym else []
    result['box_frame'] = 'model'
    result['axis_frame'] = None
    result['sym_error'] = keypoint_symmetry_error(result['keypoints'], transforms)
    if AXIS_BOX and sym and sym[0] is not None:
        p, Q, _, aligned = sym
        axis_result = _process_points(obj_id, (points - p) @ Q, aligned)
        if _valid(axis_result):
            for key in ('ply_points', 'keypoints', 'upper_kpts', 'lower_kpts'):
                axis_result[key] = axis_result[key] @ Q.T + p
            axis_error = keypoint_symmetry_error(axis_result['keypoints'], transforms)
            if axis_error < result['sym_error'] - 0.1:
                axis_result['box_frame'] = 'axis'
                axis_result['axis_frame'] = (p, Q)
                axis_result['sym_error'] = axis_error
                return axis_result
    return result

def _process_points(obj_id, points, aligned=False):
    corners = compute_2d_minimal_bbox(points) if not aligned else compute_aligned_bbox(points)
    if corners is None:
        return None
    keypoints_32 = generate_keypoints_32(corners)
    
    if obj_id not in SPLIT_CONFIG:
        return None
    
    config = SPLIT_CONFIG[obj_id]
    if config['type'] == 'z_zero':
        upper_mask, lower_mask = split_by_z_zero(keypoints_32)
    else:
        upper_mask, lower_mask = split_by_custom_line(
            keypoints_32, config['point1'], config['point2'])
    
    upper_16, upper_corners = rearrange_with_corners(keypoints_32, upper_mask)
    lower_16, lower_corners = rearrange_with_corners(keypoints_32, lower_mask)
    keypoints_final = np.vstack([upper_16, lower_16])
    
    # Edges mask 생성 (override 체크)
    if obj_id in EDGE_OVERRIDE:
        edges_mask = create_manual_edges_mask(
            upper_16, lower_16, upper_corners, lower_corners, EDGE_OVERRIDE[obj_id])
    else:
        edges_mask = create_edges_mask_cross_ratio(
            upper_16, lower_16, upper_corners, lower_corners)
    
    # Cross-ratio 검증
    cr_errors = []
    for edge_name, edge_indices in edges_mask.items():
        if len(edge_indices) == 4:
            points_4 = keypoints_final[edge_indices]
            cr = compute_cross_ratio(points_4)
            if cr is not None:
                error = abs(cr - 4.0/3.0)
                cr_errors.append(error)
    
    avg_cr_error = np.mean(cr_errors) if cr_errors else 0.0
    
    return {
        'ply_points': points,
        'keypoints': keypoints_final,
        'upper_kpts': upper_16,
        'lower_kpts': lower_16,
        'edges_mask': edges_mask,
        'upper_corners': upper_corners,
        'lower_corners': lower_corners,
        'method': config['type'],
        'avg_cr_error': avg_cr_error
    }

# ==================== 9. JSON 저장 ====================
def save_keypoints_to_json(ply_dir, output_path, obj_ids=None):
    """
    모든 객체의 keypoints와 edges_mask를 JSON으로 저장
    
    Args:
        ply_dir: PLY 파일들이 있는 디렉토리
        output_path: 저장할 JSON 파일 경로
        obj_ids: 처리할 객체 ID 리스트 (None이면 1-21 전체)
    """
    if obj_ids is None:
        obj_ids = list(range(1, 22))
    
    # 결과 저장용
    keypoints_3d = {}
    edges_mask_dict = {}
    generated_classes = []
    
    print(f"Processing {len(obj_ids)} objects...")
    
    for obj_id in obj_ids:
        ply_path = Path(ply_dir) / f"obj_{obj_id:06d}.ply"
        
        if not ply_path.exists():
            print(f"  [WARNING] Object {obj_id}: PLY file not found")
            continue
        
        result = process_single_object(obj_id, ply_path)
        
        if result is None:
            print(f"  [ERROR] Object {obj_id}: Processing failed")
            continue
        
        keypoints = result['keypoints']
        edges_mask = result['edges_mask']
        
        # keypoints를 리스트로 변환 (32개)
        keypoints_3d[str(obj_id)] = keypoints.tolist()
        
        # edges_mask를 리스트로 변환 (12개 edges, 각 4개 포인트)
        edges_list = []
        for edge_name in sorted(edges_mask.keys()):
            edge_indices = edges_mask[edge_name]
            edges_list.append(list(edge_indices))
        
        edges_mask_dict[str(obj_id)] = edges_list
        generated_classes.append(obj_id)
        
        print(f"  [OK] Object {obj_id}: {len(keypoints)} keypoints, {len(edges_list)} edges, CR error: {result['avg_cr_error']:.4f}")
    
    # JSON 구조 생성
    output_data = {
        "metadata": {
            "description": "3D keypoints with structured edges for YCB-V objects",
            "generation_method": "XY Rotating Calipers + Z MinMax + Structured Edges",
            "num_keypoints_per_class": "32 (8 corners + 24 edge midpoints)",
            "coordinate_system": "object_model_frame",
            "cross_ratio": "4/3 (exact)",
            "precision": "float64",
            "edge_structure": {
                "edges_per_object": 12,
                "points_per_edge": 4,
                "edge_types": [
                    "bottom_edges (4)",
                    "top_edges (4)",
                    "vertical_edges (4)"
                ],
                "point_order": "[corner, mid1, mid2, corner]"
            },
            "generated_classes": sorted(generated_classes)
        },
        "keypoints_3d": keypoints_3d,
        "edges_mask": edges_mask_dict
    }
    
    # JSON 파일 저장
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, 'w') as f:
        json.dump(output_data, f, indent=4)
    
    print(f"\n{'='*60}")
    print(f"Successfully saved to: {output_path}")
    print(f"Total objects: {len(generated_classes)}")
    print(f"{'='*60}\n")
    
    return output_path

# ==================== 10. 시각화 ====================
def visualize_all_objects(ply_dir, obj_ids, dataset_name='YCB-V'):
    n_objects = len(obj_ids)
    n_cols = 4
    n_rows = (n_objects + n_cols - 1) // n_cols

    # Jupyter는 작은 이미지를 확대하지 않으므로 figsize를 셀 폭보다 크게 잡아
    # 브라우저가 셀 폭에 맞춰 축소 렌더하도록 함
    SCALE = 1.4
    cell_in = 5 * SCALE
    # 1행만 있으면 figure가 너무 납작해져 여백 비율이 어색해짐 → 최소 높이 보장
    fig_h = max(cell_in * n_rows, 6 * SCALE)
    fig = plt.figure(figsize=(cell_in * n_cols, fig_h))
    _title_jobs = []
    
    for idx, obj_id in enumerate(obj_ids):
        ax = fig.add_subplot(n_rows, n_cols, idx+1, projection='3d')
        
        ply_path = Path(ply_dir) / f"obj_{obj_id:06d}.ply"
        if not ply_path.exists():
            ax.text(0.5, 0.5, 0.5, f'Obj {obj_id}\nNot found', ha='center', va='center')
            ax.set_title(f'Object {obj_id}')
            continue
        
        result = process_single_object(obj_id, ply_path)
        if result is None:
            ax.text(0.5, 0.5, 0.5, f'Obj {obj_id}\nFailed', ha='center', va='center')
            ax.set_title(f'Object {obj_id}')
            continue
        
        points = result['ply_points']
        keypoints = result['keypoints']
        upper_kpts = result['upper_kpts']
        lower_kpts = result['lower_kpts']
        edges_mask = result['edges_mask']
        avg_cr_error = result['avg_cr_error']
        
        # PLY
        ax.scatter(points[:, 0], points[:, 1], points[:, 2],
                  c='blue', s=1, alpha=0.3)
        
        # Upper - 4개 색으로 나누기 (4개씩)
        # 상부 그룹 1 (0-3): 빨간색
        ax.scatter(upper_kpts[0:4, 0], upper_kpts[0:4, 1], upper_kpts[0:4, 2],
                  c='red', s=50, marker='o', alpha=0.9, 
                  edgecolors='darkred', linewidths=1.5, label='Upper 1')
        # 상부 그룹 2 (4-7): 주황색
        ax.scatter(upper_kpts[4:16, 0], upper_kpts[4:16, 1], upper_kpts[4:16, 2],
                  c='orange', s=50, marker='o', alpha=0.9,
                  edgecolors='darkorange', linewidths=1.5, label='Upper 2')
        
        # Lower - 4개 색으로 나누기 (4개씩)
        # 하부 그룹 1 (0-3): 파란색
        ax.scatter(lower_kpts[0:4, 0], lower_kpts[0:4, 1], lower_kpts[0:4, 2],
                  c='blue', s=50, marker='s', alpha=0.9,
                  edgecolors='darkblue', linewidths=1.5, label='Lower 1')
        # 하부 그룹 2 (4-7): 하늘색
        ax.scatter(lower_kpts[4:16, 0], lower_kpts[4:16, 1], lower_kpts[4:16, 2],
                  c='cyan', s=50, marker='s', alpha=0.9,
                  edgecolors='darkcyan', linewidths=1.5, label='Lower 2')
        # Edges
        edge_colors = cm.tab20(np.linspace(0, 1, 12))
        for edge_idx, (edge_name, edge_indices) in enumerate(edges_mask.items()):
            if edge_idx >= 12:
                break
            # color = edge_colors[edge_idx]

            edge_points = keypoints[edge_indices]
            ax.plot(edge_points[:, 0], edge_points[:, 1], edge_points[:, 2],
                   color='red', linewidth=2, alpha=0.7)
        
        # 축 + 최외각 상자: 대칭축 (검은 점선)
        axis_line = np.empty((0, 3))
        if result.get('axis_frame') is not None:
            p, Q = result['axis_frame']
            h = (keypoints - p) @ Q[:, 2]
            axis_line = p + np.outer([h.min() * 1.3, h.max() * 1.3], Q[:, 2])
            ax.plot(axis_line[:, 0], axis_line[:, 1], axis_line[:, 2], color='black', linewidth=2.5, linestyle='--')

        ax.set_xlabel('X', fontsize=12)
        ax.set_ylabel('Y', fontsize=12)
        ax.set_zlabel('Z', fontsize=12)
        
        # 축 비율
        all_points = np.vstack([points, keypoints, axis_line])
        x_range = all_points[:, 0].max() - all_points[:, 0].min()
        y_range = all_points[:, 1].max() - all_points[:, 1].min()
        z_range = all_points[:, 2].max() - all_points[:, 2].min()
        max_range = max(x_range, y_range, z_range)
        if max_range > 0:
            z_ratio = max(z_range/max_range, 0.3)
            ax.set_box_aspect([x_range/max_range, y_range/max_range, z_ratio])

        # 첫 번째 물체의 스펙 저장 — 빈 슬롯을 동일 레이아웃으로 채우기 위함
        if idx == 0:
            _template_box_aspect = [x_range/max_range, y_range/max_range, z_range/max_range] if max_range > 0 else [1, 1, 1]
            _template_xlim = ax.get_xlim()
            _template_ylim = ax.get_ylim()
            _template_zlim = ax.get_zlim()
        
        # Title은 레이아웃 후 투영된 2D 높이에 맞춰 배치 (아래에서 처리)
        title = (f'Obj {obj_id} ({len(edges_mask)} edges)\nCR err: {avg_cr_error:.4f}'
                 f'\n{result["box_frame"]} box, sym err {result["sym_error"]:.1f} mm')
        _title_jobs.append((ax, title, all_points))

    # 남는 슬롯: 첫 번째 물체와 동일한 box_aspect/축 범위/라벨을 유지해
    # tight_layout이 동일 subplot으로 인식하도록 함 (내용은 그리지 않음)
    for idx in range(len(obj_ids), n_rows * n_cols):
        ax_blank = fig.add_subplot(n_rows, n_cols, idx + 1, projection='3d')
        ax_blank.set_box_aspect(_template_box_aspect)
        ax_blank.set_xlim(_template_xlim)
        ax_blank.set_ylim(_template_ylim)
        ax_blank.set_zlim(_template_zlim)
        ax_blank.set_xlabel('X', fontsize=12)
        ax_blank.set_ylabel('Y', fontsize=12)
        ax_blank.set_zlabel('Z', fontsize=12)

    # suptitle과 subplot 최상단 사이 거리를 figure 높이와 무관하게 인치 단위로 고정
    suptitle_gap_in = 2.5 * SCALE   # suptitle과 subplot 사이 여백 (inch)
    suptitle_pos_in = 1.0 * SCALE   # figure 상단에서 suptitle 중심까지 (inch)
    top_frac = 1 - suptitle_gap_in / fig_h
    suptitle_y = 1 - suptitle_pos_in / fig_h
    plt.suptitle(f'{dataset_name}: Cross-ratio Validated Edges',
                 fontsize=20, fontweight='bold', y=suptitle_y)
    plt.tight_layout(h_pad=4.0, w_pad=2.0)
    plt.subplots_adjust(left=0.08, right=0.92, top=top_frac, hspace=0.35)
    
    # 각 subplot의 실제 투영된 2D 높이에 맞춰 제목 배치
    from mpl_toolkits.mplot3d import proj3d
    fig.canvas.draw()
    for _ax, _title, _pts in _title_jobs:
        _x2, _y2, _ = proj3d.proj_transform(_pts[:, 0], _pts[:, 1], _pts[:, 2],
                                             _ax.get_proj())
        _disp = _ax.transData.transform(np.column_stack([_x2, _y2]))
        _bb = _ax.get_window_extent()
        _y_top = (_disp[:, 1].max() - _bb.y0) / _bb.height
        _ax.set_title(_title, fontsize=13, fontweight='bold', y=_y_top + 0.2)
    return fig

def debug_single_object(obj_id, ply_dir, save_path=None):
    """
    개별 물체의 상세 시각화 및 디버깅 정보 출력
    
    Args:
        obj_id: 물체 ID (1-21)
        ply_dir: PLY 파일 디렉토리
        save_path: 저장 경로 (None이면 화면에만 표시)
    """
    ply_path = Path(ply_dir) / f"obj_{obj_id:06d}.ply"
    
    if not ply_path.exists():
        print(f"[ERROR] PLY file not found: {ply_path}")
        return
    
    result = process_single_object(obj_id, ply_path)
    if result is None:
        print(f"[ERROR] Failed to process object {obj_id}")
        return
    
    points = result['ply_points']
    keypoints = result['keypoints']
    upper_kpts = result['upper_kpts']
    lower_kpts = result['lower_kpts']
    edges_mask = result['edges_mask']
    upper_corners = result['upper_corners']
    lower_corners = result['lower_corners']
    avg_cr_error = result['avg_cr_error']
    
    # 분할 정보 가져오기
    config = SPLIT_CONFIG[obj_id]
    split_type = config['type']
    
    # ==================== 출력 정보 ====================
    print(f"\n{'='*80}")
    print(f"OBJECT {obj_id} - DETAILED DEBUG INFO")
    print(f"{'='*80}")
    print(f"Split method: {split_type}")
    if split_type == 'custom_line':
        print(f"  Split line: {config['point1']} -> {config['point2']}")
    else:
        print(f"  Split plane: Z = 0")
    print(f"Average CR error: {avg_cr_error:.6f}")
    
    print(f"\n[UPPER PART]")
    print(f"  Total points: 16")
    print(f"  Corner indices: {upper_corners} ({len(upper_corners)} corners)")
    print(f"  Non-corner points: {16 - len(upper_corners)}")
    
    print(f"\n[LOWER PART]")
    print(f"  Total points: 16")
    print(f"  Corner indices: {lower_corners} ({len(lower_corners)} corners)")
    print(f"  Non-corner points: {16 - len(lower_corners)}")
    
    # Edge 정보
    print(f"\n[EDGES INFO] - Total: {len(edges_mask)} edges")
    edge_colors = cm.tab20(np.linspace(0, 1, 12))
    
    for edge_idx, (edge_name, edge_indices) in enumerate(sorted(edges_mask.items())):
        edge_points = keypoints[edge_indices]
        cr = compute_cross_ratio(edge_points)
        cr_error = abs(cr - 4.0/3.0) if cr else float('inf')
        
        # Corner 여부 확인
        is_upper = all(idx < 16 for idx in edge_indices)
        is_lower = all(idx >= 16 for idx in edge_indices)
        
        if is_upper:
            edge_type = "UPPER"
            local_indices = edge_indices
        elif is_lower:
            edge_type = "LOWER"
            local_indices = [idx - 16 for idx in edge_indices]
        else:
            edge_type = "VERTICAL"
            local_indices = edge_indices
        
        color_rgb = edge_colors[edge_idx][:3]
        color_hex = '#{:02x}{:02x}{:02x}'.format(
            int(color_rgb[0]*255), int(color_rgb[1]*255), int(color_rgb[2]*255))
        
        print(f"\n  Edge #{edge_idx+1}: {edge_name} ({edge_type})")
        print(f"    Color: {color_hex}")
        print(f"    Indices: {list(edge_indices)}")
        if edge_type != "VERTICAL":
            print(f"    Local indices: {list(local_indices)}")
        print(f"    Cross-ratio: {cr:.6f} (target: 1.333333)")
        print(f"    CR error: {cr_error:.6f}")
        
        # 길이 정보
        lengths = []
        for i in range(len(edge_indices)-1):
            p1 = keypoints[edge_indices[i]]
            p2 = keypoints[edge_indices[i+1]]
            length = np.linalg.norm(p2 - p1)
            lengths.append(length)
        print(f"    Segment lengths: {[f'{l:.2f}' for l in lengths]}")
    
    # EDGE_OVERRIDE 생성 가이드
    print(f"\n{'='*80}")
    print(f"EDGE_OVERRIDE CONFIGURATION GUIDE")
    print(f"{'='*80}")
    print(f"EDGE_OVERRIDE[{obj_id}] = {{")
    
    # Upper edges 추출
    upper_edge_pairs = []
    for edge_name, edge_indices in sorted(edges_mask.items()):
        if 'upper_edge' in edge_name:
            if all(idx < 16 for idx in edge_indices):
                c1, c2 = edge_indices[0], edge_indices[-1]
                if c1 in upper_corners and c2 in upper_corners:
                    c1_local = upper_corners.index(c1)
                    c2_local = upper_corners.index(c2)
                    upper_edge_pairs.append((c1_local, c2_local))
    
    if upper_edge_pairs:
        print(f"    'upper_edges': [")
        for pair in upper_edge_pairs:
            print(f"        {pair},  # U{pair[0]} -> U{pair[1]}")
        print(f"    ],")
    
    # Lower edges 추출
    lower_edge_pairs = []
    for edge_name, edge_indices in sorted(edges_mask.items()):
        if 'lower_edge' in edge_name:
            if all(idx >= 16 for idx in edge_indices):
                c1, c2 = edge_indices[0], edge_indices[-1]
                c1_local = c1 - 16
                c2_local = c2 - 16
                if c1_local in lower_corners and c2_local in lower_corners:
                    c1_idx = lower_corners.index(c1_local)
                    c2_idx = lower_corners.index(c2_local)
                    lower_edge_pairs.append((c1_idx, c2_idx))
    
    if lower_edge_pairs:
        print(f"    'lower_edges': [")
        for pair in lower_edge_pairs:
            print(f"        {pair},  # L{pair[0]} -> L{pair[1]}")
        print(f"    ],")
    
    # Vertical edges 추출
    vert_edge_pairs = []
    for edge_name, edge_indices in sorted(edges_mask.items()):
        if 'vertical_edge' in edge_name:
            c1, c2 = edge_indices[0], edge_indices[-1]
            if c1 < 16 and c2 >= 16:
                c1_local = c1
                c2_local = c2 - 16
                if c1_local in upper_corners and c2_local in lower_corners:
                    u_idx = upper_corners.index(c1_local)
                    l_idx = lower_corners.index(c2_local)
                    vert_edge_pairs.append((u_idx, l_idx))
    
    if vert_edge_pairs:
        print(f"    'vertical_edges': [")
        for pair in vert_edge_pairs:
            print(f"        {pair},  # U{pair[0]} -> L{pair[1]}")
        print(f"    ],")

    print(f"{'='*80}\n")
    
    # ==================== 시각화 ====================
    fig = plt.figure(figsize=(20, 12))
    
    # 1. 3D 전체 뷰
    ax1 = fig.add_subplot(2, 3, 1, projection='3d')
    ax1.scatter(points[:, 0], points[:, 1], points[:, 2],
               c='blue', s=1, alpha=0.2, label='PLY')
    
    # Upper keypoints
    ax1.scatter(upper_kpts[:, 0], upper_kpts[:, 1], upper_kpts[:, 2],
               c='red', s=100, marker='o', alpha=0.8, 
               edgecolors='darkred', linewidths=2, label='Upper')
    
    # Lower keypoints
    ax1.scatter(lower_kpts[:, 0], lower_kpts[:, 1], lower_kpts[:, 2],
               c='blue', s=100, marker='s', alpha=0.8,
               edgecolors='darkblue', linewidths=2, label='Lower')
    
    # Corners 텍스트 (Upper)
    for i, corner_idx in enumerate(upper_corners):
        pos = upper_kpts[corner_idx]
        ax1.text(pos[0], pos[1], pos[2], f'U{i}', 
                fontsize=10, fontweight='bold', color='darkred',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    # Corners 텍스트 (Lower)
    for i, corner_idx in enumerate(lower_corners):
        pos = lower_kpts[corner_idx]
        ax1.text(pos[0], pos[1], pos[2], f'L{i}', 
                fontsize=10, fontweight='bold', color='darkblue',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='cyan', alpha=0.7))
    
    # Edges with numbers
    for edge_idx, (edge_name, edge_indices) in enumerate(sorted(edges_mask.items())):
        if edge_idx >= 12:
            break
        color = edge_colors[edge_idx]
        edge_points = keypoints[edge_indices]
        ax1.plot(edge_points[:, 0], edge_points[:, 1], edge_points[:, 2],
                color=color, linewidth=3, alpha=0.8)
        
        # Edge 번호 표시 (중간점)
        mid_point = edge_points[len(edge_points)//2]
        ax1.text(mid_point[0], mid_point[1], mid_point[2], f'#{edge_idx+1}',
                fontsize=9, fontweight='bold', color='white',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.9))
    
    ax1.set_xlabel('X', fontsize=10)
    ax1.set_ylabel('Y', fontsize=10)
    ax1.set_zlabel('Z', fontsize=10)
    ax1.set_title(f'Object {obj_id} - 3D View (Edges Numbered)', fontsize=12, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=7, ncol=2)
    
    # 2. XY Top View (분할선 포함)
    ax2 = fig.add_subplot(2, 3, 2)
    ax2.scatter(points[:, 0], points[:, 1], c='blue', s=1, alpha=0.2)
    
    # Upper
    ax2.scatter(upper_kpts[:, 0], upper_kpts[:, 1],
               c='red', s=150, marker='o', alpha=0.6, 
               edgecolors='darkred', linewidths=2, label='Upper', zorder=5)
    
    # Lower
    ax2.scatter(lower_kpts[:, 0], lower_kpts[:, 1],
               c='blue', s=150, marker='s', alpha=0.6,
               edgecolors='darkblue', linewidths=2, label='Lower', zorder=5)
    
    # 분할선 그리기
    if split_type == 'custom_line':
        x1, y1 = config['point1']
        x2, y2 = config['point2']
        ax2.plot([x1, x2], [y1, y2], 'r--', linewidth=3, alpha=0.8, 
                label='Split Line', zorder=3)
        ax2.scatter([x1, x2], [y1, y2], c='red', s=200, marker='X', 
                   edgecolors='darkred', linewidths=2, zorder=4)
    else:
        # z=0 평면은 XY 평면에서는 구분이 안되므로 텍스트로만 표시
        xlim = ax2.get_xlim()
        ylim = ax2.get_ylim()
        ax2.text(0.5, 0.98, 'Split: Z = 0', transform=ax2.transAxes,
                fontsize=12, fontweight='bold', color='red',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7),
                ha='center', va='top')
    
    # Corners 번호 (Upper)
    for i, corner_idx in enumerate(upper_corners):
        pos = upper_kpts[corner_idx]
        ax2.text(pos[0], pos[1], f'U{i}', 
                fontsize=12, fontweight='bold', color='darkred',
                ha='center', va='center', zorder=10,
                bbox=dict(boxstyle='circle,pad=0.3', facecolor='yellow', alpha=0.8))
    
    # Corners 번호 (Lower)
    for i, corner_idx in enumerate(lower_corners):
        pos = lower_kpts[corner_idx]
        ax2.text(pos[0], pos[1], f'L{i}', 
                fontsize=12, fontweight='bold', color='darkblue',
                ha='center', va='center', zorder=10,
                bbox=dict(boxstyle='circle,pad=0.3', facecolor='cyan', alpha=0.8))
    
    # Edges with numbers (XY 평면)
    for edge_idx, (edge_name, edge_indices) in enumerate(sorted(edges_mask.items())):
        if edge_idx >= 12:
            break
        color = edge_colors[edge_idx]
        edge_points = keypoints[edge_indices]
        ax2.plot(edge_points[:, 0], edge_points[:, 1],
                color=color, linewidth=2.5, alpha=0.7, zorder=2)
        
        # Edge 번호 표시
        mid_point = edge_points[len(edge_points)//2]
        ax2.text(mid_point[0], mid_point[1], f'#{edge_idx+1}',
                fontsize=10, fontweight='bold', color='white',
                ha='center', va='center', zorder=11,
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.9,
                         edgecolor='black', linewidth=1.5))
    
    ax2.set_xlabel('X', fontsize=10)
    ax2.set_ylabel('Y', fontsize=10)
    ax2.set_title(f'Object {obj_id} - XY Top View (Split Line + Edge #)', 
                 fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal')
    
    # 3. XZ Side View (분할 표시)
    ax3 = fig.add_subplot(2, 3, 3)
    ax3.scatter(points[:, 0], points[:, 2], c='blue', s=1, alpha=0.2)
    ax3.scatter(upper_kpts[:, 0], upper_kpts[:, 2],
               c='red', s=150, marker='o', alpha=0.6, edgecolors='darkred', linewidths=2)
    ax3.scatter(lower_kpts[:, 0], lower_kpts[:, 2],
               c='blue', s=150, marker='s', alpha=0.6, edgecolors='darkblue', linewidths=2)
    
    # Z=0 분할선 표시
    if split_type == 'z_zero':
        xlim = ax3.get_xlim()
        ax3.axhline(y=0, color='red', linestyle='--', linewidth=3, alpha=0.8, label='Split (Z=0)')
    
    for edge_idx, (edge_name, edge_indices) in enumerate(sorted(edges_mask.items())):
        if edge_idx >= 12:
            break
        color = edge_colors[edge_idx]
        edge_points = keypoints[edge_indices]
        ax3.plot(edge_points[:, 0], edge_points[:, 2],
                color=color, linewidth=2.5, alpha=0.7)
        
        # Edge 번호
        mid_point = edge_points[len(edge_points)//2]
        ax3.text(mid_point[0], mid_point[2], f'#{edge_idx+1}',
                fontsize=8, fontweight='bold', color='white',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))
    
    ax3.set_xlabel('X', fontsize=10)
    ax3.set_ylabel('Z', fontsize=10)
    ax3.set_title(f'Object {obj_id} - XZ Side View', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.legend(fontsize=9)
    ax3.set_aspect('equal')
    
    # 4. YZ Front View (분할 표시)
    ax4 = fig.add_subplot(2, 3, 4)
    ax4.scatter(points[:, 1], points[:, 2], c='blue', s=1, alpha=0.2)
    ax4.scatter(upper_kpts[:, 1], upper_kpts[:, 2],
               c='red', s=150, marker='o', alpha=0.6, edgecolors='darkred', linewidths=2)
    ax4.scatter(lower_kpts[:, 1], lower_kpts[:, 2],
               c='blue', s=150, marker='s', alpha=0.6, edgecolors='darkblue', linewidths=2)
    
    # Z=0 분할선 표시
    if split_type == 'z_zero':
        ylim = ax4.get_ylim()
        ax4.axhline(y=0, color='red', linestyle='--', linewidth=3, alpha=0.8, label='Split (Z=0)')
    
    for edge_idx, (edge_name, edge_indices) in enumerate(sorted(edges_mask.items())):
        if edge_idx >= 12:
            break
        color = edge_colors[edge_idx]
        edge_points = keypoints[edge_indices]
        ax4.plot(edge_points[:, 1], edge_points[:, 2],
                color=color, linewidth=2.5, alpha=0.7)
        
        # Edge 번호
        mid_point = edge_points[len(edge_points)//2]
        ax4.text(mid_point[1], mid_point[2], f'#{edge_idx+1}',
                fontsize=8, fontweight='bold', color='white',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))
    
    ax4.set_xlabel('Y', fontsize=10)
    ax4.set_ylabel('Z', fontsize=10)
    ax4.set_title(f'Object {obj_id} - YZ Front View', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.legend(fontsize=9)
    ax4.set_aspect('equal')
    
    # 5. Edge 연결 정보 (텍스트)
    ax5 = fig.add_subplot(2, 3, 5)
    ax5.axis('off')
    
    info_text = f"OBJECT {obj_id} - EDGE CONNECTIONS\n"
    info_text += "="*50 + "\n"
    info_text += f"Split: {split_type}\n"
    if split_type == 'custom_line':
        info_text += f"Line: {config['point1']} -> {config['point2']}\n"
    info_text += "\n"
    
    for edge_idx, (edge_name, edge_indices) in enumerate(sorted(edges_mask.items())):
        edge_points = keypoints[edge_indices]
        cr = compute_cross_ratio(edge_points)
        cr_error = abs(cr - 4.0/3.0) if cr else float('inf')
        
        color_rgb = edge_colors[edge_idx][:3]
        
        info_text += f"Edge #{edge_idx+1} ({edge_name}):\n"
        info_text += f"  Indices: {list(edge_indices)}\n"
        info_text += f"  CR: {cr:.4f} (err: {cr_error:.4f})\n"
        
        # 길이
        lengths = []
        for i in range(len(edge_indices)-1):
            length = np.linalg.norm(
                keypoints[edge_indices[i+1]] - keypoints[edge_indices[i]])
            lengths.append(length)
        info_text += f"  Lengths: {[f'{l:.2f}' for l in lengths]}\n\n"
        
        if edge_idx >= 11:
            break
    
    ax5.text(0.05, 0.95, info_text, 
            transform=ax5.transAxes,
            fontsize=8,
            verticalalignment='top',
            fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 6. Corner 좌표 정보 + 분할 통계
    ax6 = fig.add_subplot(2, 3, 6)
    ax6.axis('off')
    
    coord_text = f"CORNER COORDINATES & SPLIT INFO\n"
    coord_text += "="*50 + "\n"
    coord_text += f"SPLIT METHOD: {split_type}\n"
    if split_type == 'custom_line':
        coord_text += f"  Point1: {config['point1']}\n"
        coord_text += f"  Point2: {config['point2']}\n"
    else:
        coord_text += f"  Plane: Z = 0\n"
    coord_text += f"\nAVG CR ERROR: {avg_cr_error:.6f}\n"
    coord_text += f"Total Edges: {len(edges_mask)}\n\n"
    
    coord_text += "UPPER CORNERS:\n"
    for i, corner_idx in enumerate(upper_corners):
        pos = upper_kpts[corner_idx]
        coord_text += f"  U{i} (idx={corner_idx}): "
        coord_text += f"({pos[0]:.2f}, {pos[1]:.2f}, {pos[2]:.2f})\n"
    
    coord_text += "\nLOWER CORNERS:\n"
    for i, corner_idx in enumerate(lower_corners):
        pos = lower_kpts[corner_idx]
        coord_text += f"  L{i} (idx={corner_idx}): "
        coord_text += f"({pos[0]:.2f}, {pos[1]:.2f}, {pos[2]:.2f})\n"
    
    ax6.text(0.05, 0.95, coord_text,
            transform=ax6.transAxes,
            fontsize=9,
            verticalalignment='top',
            fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    plt.suptitle(f'Object {obj_id} - Split Validation & Edge Numbering', 
                fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"[SAVED] Figure saved to: {save_path}")
    
    plt.show()
    
    return result

In [ ]:
if __name__ == "__main__":
     # 설정
     PLY_DIR = os.path.expanduser('~/Downloads/ycbv/models_eval')
     OUTPUT_JSON = os.path.expanduser('~/Downloads/ycbv/cached_keypoints_3d_edge.json')

     # SPLIT_CONFIG = {
     #      1: {'type': 'z_zero'}, 2: {'type': 'z_zero'}, 3: {'type': 'z_zero'},
     #      4: {'type': 'z_zero'}, 5: {'type': 'z_zero'}, 6: {'type': 'z_zero'},
     #      9: {'type': 'z_zero'}, 11: {'type': 'z_zero'}, 12: {'type': 'z_zero'},
     #      13: {'type': 'z_zero'}, 14: {'type': 'z_zero'}, 16: {'type': 'z_zero'},
     #      21: {'type': 'z_zero'},
     #      7: {'type': 'custom_line', 'point1': (60, 30), 'point2': (-60, -40)},
     #      8: {'type': 'custom_line', 'point1': (-5, 25), 'point2': (5, -25)},
     #      10: {'type': 'custom_line', 'point1': (60, -10), 'point2': (-60, 30)},
     #      15: {'type': 'custom_line', 'point1': (60, 0), 'point2': (-60, 0)},
     #      17: {'type': 'custom_line', 'point1': (-60, 30), 'point2': (60, -10)},
     #      18: {'type': 'custom_line', 'point1': (60, -10), 'point2': (-60, 30)},
     #      19: {'type': 'custom_line', 'point1': (-60, 0), 'point2': (60, 0)},
     #      20: {'type': 'custom_line', 'point1': (0, 30), 'point2': (0, -30)},
     #      }
     
     SPLIT_CONFIG = {
          1: {'type': 'z_zero'}, 2: {'type': 'z_zero'}, 3: {'type': 'z_zero'},
          4: {'type': 'z_zero'}, 5: {'type': 'z_zero'}, 6: {'type': 'z_zero'},
          9: {'type': 'z_zero'}, 11: {'type': 'z_zero'}, 12: {'type': 'z_zero'},
          13: {'type': 'z_zero'}, 14: {'type': 'z_zero'}, 16: {'type': 'z_zero'},
          21: {'type': 'z_zero'},
          7: {'type': 'z_zero'},
          8: {'type': 'z_zero'},
          10: {'type': 'z_zero'},
          15: {'type': 'custom_line', 'point1': (60, 0), 'point2': (-60, 0)},
          17: {'type': 'z_zero'},
          18: {'type': 'custom_line', 'point1': (60, -10), 'point2': (-60, 30)},
          19: {'type': 'z_zero'},
          20: {'type': 'z_zero'},
          }
    
     # Edges 수동 정의 (자동 매핑이 실패하는 객체들)
     EDGE_OVERRIDE = {
          # 15번: 수동으로 edges 정의 필요
          # 사용법: upper corners 인덱스, lower corners 인덱스 기반
          15: {'upper_edges': [(0, 1), (2, 3), (0, 2), (1, 3)],
          'lower_edges': [(0, 1), (2, 3), (0, 2), (1, 3)],
          'vertical_edges': [(0, 1), (1, 0), (2, 3), (3, 2)]},

          # 18번: 수동으로 edges 정의 필요  
          18: {'upper_edges': [(0, 2), (0, 1), (1, 3), (2, 3)],
          'lower_edges': [(0, 2), (0, 1), (1, 3), (2, 3)],
          'vertical_edges': [(0,1), (1,0), (2,3), (3,2)]},

          }

     # YCB-Video 21개 객체 (1, 2, 3, ..., 21)
     OBJ_IDS = list(range(1, 22))

     # 전체 시각화
     # fig = visualize_all_objects(ply_dir=PLY_DIR, obj_ids=OBJ_IDS, dataset_name='YCB-V')
     # plt.show()

     # # JSON 저장
     saved_path = save_keypoints_to_json(PLY_DIR, OUTPUT_JSON, OBJ_IDS)

     # 특정 객체 디버깅 (15번, 18번 등)
     # debug_single_object(15, PLY_DIR)
     # for i in [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21]:
     #     debug_single_object(i, ply_dir)

In [ ]:
if __name__ == "__main__":
     # 설정
     PLY_DIR = os.path.expanduser('~/Downloads/lmo/models_eval')
     OUTPUT_JSON = os.path.expanduser("~/Downloads/lmo/cached_keypoints_3d_edge.json")

     SPLIT_CONFIG = {
          1: {'type': 'z_zero'}, 2: {'type': 'z_zero'}, 3: {'type': 'z_zero'},
          4: {'type': 'z_zero'}, 5: {'type': 'z_zero'}, 6: {'type': 'z_zero'},
          7: {'type': 'z_zero'}, 8: {'type': 'z_zero'}, 9: {'type': 'z_zero'},
          10: {'type': 'z_zero'}, 11: {'type': 'z_zero'}, 12: {'type': 'z_zero'},
          13: {'type': 'z_zero'}, 14: {'type': 'z_zero'}, 15: {'type': 'z_zero'},
          }
    
     # Edges 수동 정의 (자동 매핑이 실패하는 객체들)
     EDGE_OVERRIDE = {
          15: {
               'upper_edges': [
                    (0, 1),  # Edge 1: U0 -> U1
                    (1, 2),  # Edge 2: U1 -> U2  
                    (2, 3),  # Edge 3: U2 -> U3
                    (3, 0),  # Edge 4: U3 -> U0
               ], 
               'lower_edges': [
                    (0, 1),  # Edge 5: L0 -> L1
                    (1, 2),  # Edge 6: L1 -> L2
                    (2, 3),  # Edge 7: L2 -> L3
                    (3, 0),  # Edge 8: L3 -> L0
               ],
               'vertical_edges': [
                    (0, 0),  # Edge 9: U0 -> L0
                    (1, 1),  # Edge 10: U1 -> L1
                    (2, 2),  # Edge 11: U2 -> L2
                    (3, 3),  # Edge 12: U3 -> L3
               ]
          },
          }

     # lmo 8개 객체 (1, 5, 6, 8, 9, 10, 11, 12)
     OBJ_IDS = [1, 5, 6, 8, 9, 10, 11, 12]

     #  # 전체 시각화
     #  print("🚀 Cross-ratio 검증 시각화 시작...")
     # fig = visualize_all_objects(ply_dir=PLY_DIR, obj_ids=OBJ_IDS, dataset_name='LMO')
     # plt.show()
     # print("✅ 완료!")

     # JSON 저장
     saved_path = save_keypoints_to_json(PLY_DIR, OUTPUT_JSON, OBJ_IDS)

     #  특정 객체 디버깅 (15번, 18번 등)
     # debug_single_object(15, PLY_DIR)

In [ ]:
if __name__ == "__main__":
     # 설정
     PLY_DIR = os.path.expanduser('~/Downloads/hb/models_eval')
     OUTPUT_JSON = os.path.expanduser('~/Downloads/hb/cached_keypoints_3d_edge.json')

     SPLIT_CONFIG = {
          1: {'type': 'z_zero'}, 2: {'type': 'z_zero'}, 3: {'type': 'z_zero'},
          4: {'type': 'z_zero'}, 5: {'type': 'z_zero'}, 6: {'type': 'z_zero'},
          7: {'type': 'z_zero'}, 8: {'type': 'z_zero'}, 9: {'type': 'z_zero'},
          10: {'type': 'z_zero'}, 11: {'type': 'z_zero'}, 12: {'type': 'z_zero'},
          13: {'type': 'z_zero'}, 14: {'type': 'z_zero'}, 15: {'type': 'z_zero'},
          16: {'type': 'z_zero'}, 17: {'type': 'z_zero'}, 18: {'type': 'z_zero'},
          19: {'type': 'z_zero'}, 20: {'type': 'z_zero'}, 21: {'type': 'z_zero'},
          22: {'type': 'z_zero'}, 23: {'type': 'z_zero'}, 24: {'type': 'z_zero'},
          25: {'type': 'z_zero'}, 26: {'type': 'z_zero'}, 27: {'type': 'z_zero'},
          28: {'type': 'z_zero'}, 29: {'type': 'z_zero'}, 30: {'type': 'z_zero'},
          31: {'type': 'z_zero'}, 32: {'type': 'z_zero'}, 33: {'type': 'z_zero'},
          }
    
     # Edges 수동 정의 (자동 매핑이 실패하는 객체들)
     EDGE_OVERRIDE = {
          15: {
               'upper_edges': [
                    (0, 1),  # Edge 1: U0 -> U1
                    (1, 2),  # Edge 2: U1 -> U2  
                    (2, 3),  # Edge 3: U2 -> U3
                    (3, 0),  # Edge 4: U3 -> U0
               ], 
               'lower_edges': [
                    (0, 1),  # Edge 5: L0 -> L1
                    (1, 2),  # Edge 6: L1 -> L2
                    (2, 3),  # Edge 7: L2 -> L3
                    (3, 0),  # Edge 8: L3 -> L0
               ],
               'vertical_edges': [
                    (0, 0),  # Edge 9: U0 -> L0
                    (1, 1),  # Edge 10: U1 -> L1
                    (2, 2),  # Edge 11: U2 -> L2
                    (3, 3),  # Edge 12: U3 -> L3
               ]
          },
          
          18: {
               'upper_edges': [(0, 1), (1, 2), (2, 3), (3, 0)],
               'lower_edges': [(0, 1), (1, 2), (2, 3), (3, 0)],
               'vertical_edges': [(0, 0), (1, 1), (2, 2), (3, 3)]
          },
          }

     # hb 33개 객체 (1, 2, 3, ..., 33)
     OBJ_IDS = list(range(1, 34))

     #  # 전체 시각화
     #  print("🚀 Cross-ratio 검증 시각화 시작...")
     # fig = visualize_all_objects(ply_dir=PLY_DIR, obj_ids=OBJ_IDS, dataset_name='HB')
     # plt.show()
     # print("✅ 완료!")

     # JSON 저장
     saved_path = save_keypoints_to_json(PLY_DIR, OUTPUT_JSON, OBJ_IDS)

     #  특정 객체 디버깅 (15번, 18번 등)
     # debug_single_object(15, PLY_DIR)

In [ ]:
if __name__ == "__main__":
     # 설정
     PLY_DIR = os.path.expanduser('~/Downloads/tudl/models_eval')
     OUTPUT_JSON = os.path.expanduser('~/Downloads/tudl/cached_keypoints_3d_edge.json')

     SPLIT_CONFIG = {
          1: {'type': 'z_zero'}, 2: {'type': 'z_zero'}, 3: {'type': 'z_zero'},
          }
    
     # Edges 수동 정의 (자동 매핑이 실패하는 객체들)
    #  EDGE_OVERRIDE = {
    #       15: {
    #            'upper_edges': [
    #                 (0, 1),  # Edge 1: U0 -> U1
    #                 (1, 2),  # Edge 2: U1 -> U2  
    #                 (2, 3),  # Edge 3: U2 -> U3
    #                 (3, 0),  # Edge 4: U3 -> U0
    #            ], 
    #            'lower_edges': [
    #                 (0, 1),  # Edge 5: L0 -> L1
    #                 (1, 2),  # Edge 6: L1 -> L2
    #                 (2, 3),  # Edge 7: L2 -> L3
    #                 (3, 0),  # Edge 8: L3 -> L0
    #            ],
    #            'vertical_edges': [
    #                 (0, 0),  # Edge 9: U0 -> L0
    #                 (1, 1),  # Edge 10: U1 -> L1
    #                 (2, 2),  # Edge 11: U2 -> L2
    #                 (3, 3),  # Edge 12: U3 -> L3
    #            ]
    #       },
          
    #       18: {
    #            'upper_edges': [(0, 1), (1, 2), (2, 3), (3, 0)],
    #            'lower_edges': [(0, 1), (1, 2), (2, 3), (3, 0)],
    #            'vertical_edges': [(0, 0), (1, 1), (2, 2), (3, 3)]
    #       },
    #       }

     EDGE_OVERRIDE = {}

     # hb 33개 객체 (1, 2, 3, ..., 33)
     OBJ_IDS = list(range(1, 4))

     #  # 전체 시각화
     #  print("🚀 Cross-ratio 검증 시각화 시작...")
     # fig = visualize_all_objects(ply_dir=PLY_DIR, obj_ids=OBJ_IDS, dataset_name='TUDL')
     # plt.show()
     # print("✅ 완료!")

     # JSON 저장
     saved_path = save_keypoints_to_json(PLY_DIR, OUTPUT_JSON, OBJ_IDS)

     #  특정 객체 디버깅 (15번, 18번 등)
     # debug_single_object(15, PLY_DIR)

In [ ]:
if __name__ == "__main__":
     # 설정
     PLY_DIR = os.path.expanduser('~/Downloads/icbin/models_eval')
     OUTPUT_JSON = os.path.expanduser('~/Downloads/icbin/cached_keypoints_3d_edge.json')

     SPLIT_CONFIG = {
          1: {'type': 'z_zero'}, 2: {'type': 'z_zero'},
          }
    
     # Edges 수동 정의 (자동 매핑이 실패하는 객체들)
    #  EDGE_OVERRIDE = {
    #       15: {
    #            'upper_edges': [
    #                 (0, 1),  # Edge 1: U0 -> U1
    #                 (1, 2),  # Edge 2: U1 -> U2  
    #                 (2, 3),  # Edge 3: U2 -> U3
    #                 (3, 0),  # Edge 4: U3 -> U0
    #            ], 
    #            'lower_edges': [
    #                 (0, 1),  # Edge 5: L0 -> L1
    #                 (1, 2),  # Edge 6: L1 -> L2
    #                 (2, 3),  # Edge 7: L2 -> L3
    #                 (3, 0),  # Edge 8: L3 -> L0
    #            ],
    #            'vertical_edges': [
    #                 (0, 0),  # Edge 9: U0 -> L0
    #                 (1, 1),  # Edge 10: U1 -> L1
    #                 (2, 2),  # Edge 11: U2 -> L2
    #                 (3, 3),  # Edge 12: U3 -> L3
    #            ]
    #       },
          
    #       18: {
    #            'upper_edges': [(0, 1), (1, 2), (2, 3), (3, 0)],
    #            'lower_edges': [(0, 1), (1, 2), (2, 3), (3, 0)],
    #            'vertical_edges': [(0, 0), (1, 1), (2, 2), (3, 3)]
    #       },
    #       }

     EDGE_OVERRIDE = {}

     # hb 33개 객체 (1, 2, 3, ..., 33)
     OBJ_IDS = list(range(1, 3))

     #  # 전체 시각화
     #  print("🚀 Cross-ratio 검증 시각화 시작...")
     # fig = visualize_all_objects(ply_dir=PLY_DIR, obj_ids=OBJ_IDS, dataset_name='ICBIN')
     # plt.show()
     # print("✅ 완료!")

     # JSON 저장
     saved_path = save_keypoints_to_json(PLY_DIR, OUTPUT_JSON, OBJ_IDS)

     #  특정 객체 디버깅 (15번, 18번 등)
     # debug_single_object(15, PLY_DIR)

In [ ]:
if __name__ == "__main__":
     # 설정
     PLY_DIR = os.path.expanduser('~/Downloads/itodd/models_eval')
     OUTPUT_JSON = os.path.expanduser('~/Downloads/itodd/cached_keypoints_3d_edge.json')

     SPLIT_CONFIG = {
          1: {'type': 'z_zero'}, 2: {'type': 'z_zero'}, 3: {'type': 'z_zero'},
          4: {'type': 'z_zero'}, 5: {'type': 'z_zero'}, 6: {'type': 'z_zero'},
          7: {'type': 'z_zero'}, 8: {'type': 'z_zero'}, 9: {'type': 'z_zero'},
          10: {'type': 'z_zero'}, 11: {'type': 'z_zero'}, 12: {'type': 'z_zero'},
          13: {'type': 'z_zero'}, 14: {'type': 'z_zero'}, 15: {'type': 'z_zero'},
          16: {'type': 'z_zero'}, 17: {'type': 'z_zero'}, 18: {'type': 'z_zero'},
          19: {'type': 'z_zero'}, 20: {'type': 'z_zero'}, 21: {'type': 'z_zero'},
          22: {'type': 'z_zero'}, 23: {'type': 'z_zero'}, 24: {'type': 'z_zero'},
          25: {'type': 'z_zero'}, 26: {'type': 'z_zero'}, 27: {'type': 'z_zero'},
          28: {'type': 'z_zero'},
        #   7: {'type': 'custom_line', 'point1': (60, 30), 'point2': (-60, -40)},
        #   8: {'type': 'custom_line', 'point1': (-5, 25), 'point2': (5, -25)},
        #   10: {'type': 'custom_line', 'point1': (60, -10), 'point2': (-60, 30)},
        #   15: {'type': 'custom_line', 'point1': (60, 0), 'point2': (-60, 0)},
        #   17: {'type': 'custom_line', 'point1': (-60, 30), 'point2': (60, -10)},
        #   18: {'type': 'custom_line', 'point1': (60, -10), 'point2': (-60, 30)},
        #   19: {'type': 'custom_line', 'point1': (-60, 0), 'point2': (60, 0)},
        #   20: {'type': 'custom_line', 'point1': (0, 30), 'point2': (0, -30)},
          }
    
     # Edges 수동 정의 (자동 매핑이 실패하는 객체들)
    #  EDGE_OVERRIDE = {
    #       # 15번: 수동으로 edges 정의 필요
    #       # 사용법: upper corners 인덱스, lower corners 인덱스 기반
    #       15: {'upper_edges': [(0, 2), (0, 1), (1, 3), (2, 3)], 
    #       'lower_edges': [(0, 2), (0, 1), (1, 3), (2, 3)],
    #       'vertical_edges': [(0,3), (1,2), (2,1), (3,2)]},

    #       # 18번: 수동으로 edges 정의 필요  
    #       18: {'upper_edges': [(0, 2), (0, 1), (1, 3), (2, 3)],
    #       'lower_edges': [(0, 2), (0, 1), (1, 3), (2, 3)],
    #       'vertical_edges': [(0,1), (1,0), (2,3), (3,2)]},
    #       }

     EDGE_OVERRIDE = {}

     # ITODD 28개 객체 (1, 2, 3, ..., 28)
     OBJ_IDS = list(range(1, 29))

     # 전체 시각화
     # fig = visualize_all_objects(ply_dir=PLY_DIR, obj_ids=OBJ_IDS, dataset_name='ITODD')
     # plt.show()

     # # JSON 저장
     saved_path = save_keypoints_to_json(PLY_DIR, OUTPUT_JSON, OBJ_IDS)

     # 특정 객체 디버깅 (15번, 18번 등)
     # debug_single_object(15, ply_dir)
     # for i in [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21]:
     #     debug_single_object(i, ply_dir)

In [ ]:
if __name__ == "__main__":
     # 설정
     PLY_DIR = os.path.expanduser('~/Downloads/tless/models_eval')
     OUTPUT_JSON = os.path.expanduser('~/Downloads/tless/cached_keypoints_3d_edge.json')

     SPLIT_CONFIG = {
          1: {'type': 'z_zero'}, 2: {'type': 'z_zero'}, 3: {'type': 'z_zero'},
          4: {'type': 'z_zero'}, 5: {'type': 'z_zero'}, 6: {'type': 'z_zero'},
          7: {'type': 'z_zero'}, 8: {'type': 'z_zero'}, 9: {'type': 'z_zero'},
          10: {'type': 'z_zero'}, 11: {'type': 'z_zero'}, 12: {'type': 'z_zero'},
          13: {'type': 'z_zero'}, 14: {'type': 'z_zero'}, 15: {'type': 'z_zero'},
          16: {'type': 'z_zero'}, 17: {'type': 'z_zero'}, 18: {'type': 'z_zero'},
          19: {'type': 'z_zero'}, 20: {'type': 'z_zero'}, 21: {'type': 'z_zero'},
          22: {'type': 'z_zero'}, 23: {'type': 'z_zero'}, 24: {'type': 'z_zero'},
          25: {'type': 'z_zero'}, 26: {'type': 'z_zero'}, 27: {'type': 'z_zero'},
          28: {'type': 'z_zero'}, 29: {'type': 'z_zero'}, 30: {'type': 'z_zero'},
          }

     EDGE_OVERRIDE = {}

     # T-LESS 30개 객체 (1, 2, 3, ..., 30)
     OBJ_IDS = list(range(1, 31))

     # 전체 시각화
     # fig = visualize_all_objects(ply_dir=PLY_DIR, obj_ids=OBJ_IDS, dataset_name='T-LESS')
     # plt.show()

     # # JSON 저장
     saved_path = save_keypoints_to_json(PLY_DIR, OUTPUT_JSON, OBJ_IDS)

     # 특정 객체 디버깅
     # debug_single_object(25, PLY_DIR)